In [41]:
import pandas as pd
from utils.read import readExcel
from datetime import datetime

In [42]:
df = readExcel(
    "./raw-sheets-dump/19th July 2026.xlsx",
    "Sheet1",
)[0]

shelfLife = readExcel("./shelf-life/Summer 2026 PUNE City Shelf Life.xlsx", "Sheet1")[0]
shelfLife.columns = shelfLife.iloc[0]
shelfLife = shelfLife.iloc[1:]

nx = readExcel("./nx/NX Article SL.xlsx", "Sheet1")[0]

df.columns = df.columns.str.strip()
shelfLife.columns = shelfLife.columns.str.strip()
nx.columns = nx.columns.str.strip()

In [43]:
merged = df.merge(shelfLife, how="inner", left_on="ITEM_CODE", right_on="Article Code")

In [44]:
finalDf = merged.copy()
finalDf.columns = finalDf.columns.str.strip()
finalDf = finalDf[["PRODUCT_NAME", "WEIGHT", "Symbol", "Date", "ITEM_CODE", "Indents"]]
finalDf

,PRODUCT_NAME,WEIGHT,Symbol,Date,ITEM_CODE,Indents
0,Ash Gourd Portion,1Piece,C,2026-07-19,5949.0,1.0
1,Ash Gourd Portion,1Piece,C,2026-07-19,5949.0,1.0
2,Ash Gourd Portion,1Piece,C,2026-07-19,5949.0,1.0
3,Baby Banana,4Pieces,T,2026-07-19,11565.0,28.0
4,Baby Banana,4Pieces,T,2026-07-19,11565.0,14.0
...,...,...,...,...,...,...
512,Yellow Shevanti Flowers,100g,C,2026-07-19,9497.0,2.0
513,Yellow Shevanti Flowers,100g,C,2026-07-19,9497.0,2.0
514,Yellow Shevanti Flowers,100g,C,2026-07-19,9497.0,3.0
515,Yellow Shevanti Flowers,100g,C,2026-07-19,9497.0,2.0


In [45]:
finalDf["Date"] = pd.to_datetime(finalDf["Date"])
finalDf["Date"] = finalDf["Date"] - pd.Timedelta(days=1)

In [46]:
day = finalDf["Date"].apply(lambda x: (x.isoweekday() % 7) + 1).unique()[0] - 1
dayOfTheWeek = shelfLife.columns[8:15][day]

In [47]:
finalDf

,PRODUCT_NAME,WEIGHT,Symbol,Date,ITEM_CODE,Indents
0,Ash Gourd Portion,1Piece,C,2026-07-18,5949.0,1.0
1,Ash Gourd Portion,1Piece,C,2026-07-18,5949.0,1.0
2,Ash Gourd Portion,1Piece,C,2026-07-18,5949.0,1.0
3,Baby Banana,4Pieces,T,2026-07-18,11565.0,28.0
4,Baby Banana,4Pieces,T,2026-07-18,11565.0,14.0
...,...,...,...,...,...,...
512,Yellow Shevanti Flowers,100g,C,2026-07-18,9497.0,2.0
513,Yellow Shevanti Flowers,100g,C,2026-07-18,9497.0,2.0
514,Yellow Shevanti Flowers,100g,C,2026-07-18,9497.0,3.0
515,Yellow Shevanti Flowers,100g,C,2026-07-18,9497.0,2.0


In [48]:
finalDf["Day"] = finalDf.merge(
    shelfLife, how="inner", left_on="ITEM_CODE", right_on="Article Code"
)[dayOfTheWeek]

In [51]:
finalDf["ITEM_CODE"] = finalDf["ITEM_CODE"].astype(int)
nx["ITEM_CODE"] = nx["ITEM_CODE"].astype(int)
finalDf["Day"] = finalDf["Day"].astype(int)

In [52]:
finalDf["ID"] = finalDf.apply(
    lambda row: "b_"
    + str(row["ITEM_CODE"])
    + "_"
    + (row["Date"] + pd.Timedelta(days=1)).strftime("%d-%m-%Y"),
    axis=1,
)

In [53]:
finalDf["NX"] = ""
for i in nx["ITEM_CODE"]:
    finalDf.loc[finalDf["ITEM_CODE"] == i, "NX"] = "NX"

In [54]:
finalDf = finalDf.loc[finalDf.index.repeat(finalDf["Indents"])]

In [55]:
finalDf

,PRODUCT_NAME,WEIGHT,Symbol,Date,ITEM_CODE,Indents,Day,ID,NX
0,Ash Gourd Portion,1Piece,C,2026-07-18,5949,1.0,1,b_5949_19-07-2026,
1,Ash Gourd Portion,1Piece,C,2026-07-18,5949,1.0,1,b_5949_19-07-2026,
2,Ash Gourd Portion,1Piece,C,2026-07-18,5949,1.0,1,b_5949_19-07-2026,
3,Baby Banana,4Pieces,T,2026-07-18,11565,28.0,2,b_11565_19-07-2026,NX
3,Baby Banana,4Pieces,T,2026-07-18,11565,28.0,2,b_11565_19-07-2026,NX
...,...,...,...,...,...,...,...,...,...
514,Yellow Shevanti Flowers,100g,C,2026-07-18,9497,3.0,1,b_9497_19-07-2026,
515,Yellow Shevanti Flowers,100g,C,2026-07-18,9497,2.0,1,b_9497_19-07-2026,
515,Yellow Shevanti Flowers,100g,C,2026-07-18,9497,2.0,1,b_9497_19-07-2026,
516,Yellow Shevanti Flowers,100g,C,2026-07-18,9497,2.0,1,b_9497_19-07-2026,
